# Project 3 — Food & Restaurant Services: AI Demand Forecasting
# Week 3 and Week 4 end-to-end pipeline using the provided CSV datasets

"""
Build an end-to-end weekly demand forecasting pipeline for restaurant operations using the
provided local dataset files:
- train.csv
- test.csv
- meal_info.csv
- fulfilment_center_info.csv

The notebook cleans the data, engineers week-based features, trains baseline and advanced models,
evaluates performance, creates multi-step forecasts, and saves artifacts for submission.
"""

In [ ]:
# ==============================
# 1) Install & Import Dependencies
# ==============================

# Imports only; install missing packages separately if needed in the local environment.
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception:
    XGBRegressor = None
    HAS_XGBOOST = False

print('Imports complete; xgboost available =', HAS_XGBOOST)

In [ ]:
# ==============================
# 2) Load Local Dataset Files
# ==============================

from pathlib import Path

# Search the current workspace, nearby parents, and the workspace drive for the provided files.
candidate_roots = []
cwd = Path.cwd().resolve()
candidate_roots.append(cwd)
candidate_roots.extend(list(cwd.parents))
candidate_roots.append(Path.home())

workspace_drive = Path('E:/')
if workspace_drive.exists():
    candidate_roots.append(workspace_drive)

search_roots = []
for root in candidate_roots:
    if root.exists() and root not in search_roots:
        search_roots.append(root)

def find_file(filename):
    for root in search_roots:
        for current_root, _, files in os.walk(str(root)):
            if filename in files:
                return os.path.join(current_root, filename)
    return None

train_path = find_file('train.csv')
test_path = find_file('test.csv')
meal_path = find_file('meal_info.csv')
center_path = find_file('fulfilment_center_info.csv')
sample_path = find_file('sample_submission.csv')

for file_path in [train_path, test_path, meal_path, center_path]:
    print(file_path, '->', bool(file_path and os.path.exists(file_path)))

if not all([train_path, test_path, meal_path, center_path]):
    raise FileNotFoundError('Could not locate all dataset CSV files in the notebook runtime.')

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
meal_info = pd.read_csv(meal_path)
center_info = pd.read_csv(center_path)
sample_submission = pd.read_csv(sample_path) if sample_path else pd.DataFrame()

print('train_raw shape:', train_raw.shape)
print('test_raw shape:', test_raw.shape)
print('meal_info shape:', meal_info.shape)
print('center_info shape:', center_info.shape)
if not sample_submission.empty:
    print('sample_submission shape:', sample_submission.shape)

train_raw.head()

In [ ]:
# ==============================
# 3) Clean, Merge, and Inspect the Weekly Data
# ==============================

# Standardize column names
train_raw.columns = train_raw.columns.str.strip().str.lower()
test_raw.columns = test_raw.columns.str.strip().str.lower()
meal_info.columns = meal_info.columns.str.strip().str.lower()
center_info.columns = center_info.columns.str.strip().str.lower()

# Expected schema checks
required_train_cols = {'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'num_orders'}
required_test_cols = {'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured'}

missing_train = required_train_cols - set(train_raw.columns)
missing_test = required_test_cols - set(test_raw.columns)
if missing_train:
    raise ValueError(f'Train file is missing columns: {sorted(missing_train)}')
if missing_test:
    raise ValueError(f'Test file is missing columns: {sorted(missing_test)}')

# Remove duplicates and coerce numeric columns
for frame in [train_raw, test_raw, meal_info, center_info]:
    frame.drop_duplicates(inplace=True)

numeric_cols = ['week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured']
for frame in [train_raw, test_raw]:
    for column in numeric_cols:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')

train_raw['num_orders'] = pd.to_numeric(train_raw['num_orders'], errors='coerce')

# Merge metadata
train_df = train_raw.merge(meal_info, on='meal_id', how='left').merge(center_info, on='center_id', how='left')
test_df = test_raw.merge(meal_info, on='meal_id', how='left').merge(center_info, on='center_id', how='left')

# Fill obvious missing metadata values
for frame in [train_df, test_df]:
    for column in frame.columns:
        if frame[column].dtype == 'object':
            frame[column] = frame[column].fillna('Unknown')
        else:
            frame[column] = frame[column].fillna(frame[column].median())

# Sort by time and identifiers
train_df = train_df.sort_values(['center_id', 'meal_id', 'week']).reset_index(drop=True)
test_df = test_df.sort_values(['center_id', 'meal_id', 'week']).reset_index(drop=True)

print('Merged train shape:', train_df.shape)
print('Merged test shape:', test_df.shape)
print('Train columns:', list(train_df.columns))
train_df.head()

# Data cleaning status
# The notebook now loads the local CSV files directly, merges center and meal metadata,
# and prepares the weekly target for modeling.

print('Missing values in train after merge:')
print(train_df.isna().sum().sort_values(ascending=False).head(10))
print('\nTarget summary:')
print(train_df['num_orders'].describe())

In [ ]:
# ==============================
# 4) Weekly Exploratory Data Analysis
# ==============================

plt.figure(figsize=(12, 4))
sns.lineplot(data=train_df.groupby('week', as_index=False)['num_orders'].sum(), x='week', y='num_orders', color='tab:blue')
plt.title('Total Weekly Orders')
plt.xlabel('Week')
plt.ylabel('Orders')
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(10, 4))
sns.boxplot(data=train_df, x='emailer_for_promotion', y='num_orders')
plt.title('Orders by Email Promotion Flag')
plt.show()

plt.figure(figsize=(10, 4))
sns.boxplot(data=train_df, x='homepage_featured', y='num_orders')
plt.title('Orders by Homepage Featured Flag')
plt.show()

plt.figure(figsize=(12, 4))
sns.scatterplot(data=train_df.sample(min(len(train_df), 2000), random_state=42), x='checkout_price', y='num_orders', alpha=0.35)
plt.title('Checkout Price vs Orders')
plt.show()

train_df.groupby('category')['num_orders'].mean().sort_values(ascending=False).head(10)

In [ ]:
# ==============================
# 5) Create Week-Based Forecast Features
# ==============================

def add_weekly_features(df):
    df = df.copy()
    df = df.sort_values(['center_id', 'meal_id', 'week']).reset_index(drop=True)

    group_cols = ['center_id', 'meal_id']
    df['price_diff'] = df['checkout_price'] - df['base_price']
    df['price_discount_pct'] = np.where(df['base_price'] > 0, (df['base_price'] - df['checkout_price']) / df['base_price'], 0)
    df['is_discount'] = (df['price_discount_pct'] > 0).astype(int)
    df['price_ratio'] = np.where(df['base_price'] > 0, df['checkout_price'] / df['base_price'], 1)

    for lag in [1, 2, 3, 4]:
        df[f'lag_orders_{lag}'] = df.groupby(group_cols)['num_orders'].shift(lag)
        df[f'lag_checkout_price_{lag}'] = df.groupby(group_cols)['checkout_price'].shift(lag)

    df['rolling_mean_4'] = df.groupby(group_cols)['num_orders'].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
    df['rolling_std_4'] = df.groupby(group_cols)['num_orders'].transform(lambda s: s.shift(1).rolling(4, min_periods=1).std())
    df['rolling_mean_8'] = df.groupby(group_cols)['num_orders'].transform(lambda s: s.shift(1).rolling(8, min_periods=1).mean())

    df['week_sin'] = np.sin(2 * np.pi * df['week'] / 52.0)
    df['week_cos'] = np.cos(2 * np.pi * df['week'] / 52.0)
    df['center_week_avg'] = df.groupby('center_id')['num_orders'].transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
    df['meal_week_avg'] = df.groupby('meal_id')['num_orders'].transform(lambda s: s.shift(1).expanding(min_periods=1).mean())

    df = df.dropna(subset=['lag_orders_1', 'rolling_mean_4']).copy()
    return df

model_df = add_weekly_features(train_df)
MAX_MODEL_ROWS = 80000
if len(model_df) > MAX_MODEL_ROWS:
    model_df = model_df.iloc[-MAX_MODEL_ROWS:].copy()
    print(f'Training frame capped to the most recent {MAX_MODEL_ROWS} rows for faster execution.')

print('Model frame shape:', model_df.shape)
model_df.head()

In [ ]:
# ==============================
# 6) Weekly Feature Diagnostics
# ==============================

feature_snapshot = [c for c in model_df.columns if c != 'num_orders']
print('Weekly model frame shape:', model_df.shape)
print('Feature count:', len(feature_snapshot))
print('Week range:', int(model_df['week'].min()), 'to', int(model_df['week'].max()))
print('Target summary:')
print(model_df['num_orders'].describe())

numeric_preview = model_df[feature_snapshot + ['num_orders']].select_dtypes(include=[np.number]).corr(numeric_only=True)['num_orders'].sort_values(ascending=False).head(10)
print('Top numeric correlations with num_orders:')
print(numeric_preview)

In [ ]:
# ==============================
# 6) Train/Test Split (time-aware by week)
# ==============================

feature_cols = [
    'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured',
    'category', 'cuisine', 'city_code', 'region_code', 'center_type',
    'price_diff', 'price_discount_pct', 'is_discount', 'price_ratio',
    'lag_orders_1', 'lag_orders_2', 'lag_orders_3', 'lag_orders_4',
    'lag_checkout_price_1', 'lag_checkout_price_2', 'lag_checkout_price_3', 'lag_checkout_price_4',
    'rolling_mean_4', 'rolling_std_4', 'rolling_mean_8', 'week', 'week_sin', 'week_cos',
    'center_week_avg', 'meal_week_avg'
]

feature_cols = [c for c in feature_cols if c in model_df.columns]
X = model_df[feature_cols].copy()
y = model_df['num_orders'].copy()

# Hold out the latest weeks for validation
unique_weeks = np.sort(model_df['week'].unique())
validation_weeks = unique_weeks[-4:] if len(unique_weeks) >= 4 else unique_weeks[-1:]
train_mask = ~model_df['week'].isin(validation_weeks)
val_mask = model_df['week'].isin(validation_weeks)

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()
X_val = X.loc[val_mask].copy()
y_val = y.loc[val_mask].copy()

print('Train shape:', X_train.shape, 'Validation shape:', X_val.shape)
print('Validation weeks:', validation_weeks)

In [ ]:
# ==============================
# 7) Baseline Models + Tree Model
# ==============================

categorical_features = [c for c in ['category', 'cuisine', 'center_type'] if c in X_train.columns]
numeric_features = [c for c in X_train.columns if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ],
    remainder='drop'
)

lr_model = Pipeline([
    ('prep', preprocessor),
    ('model', LinearRegression())
])

rf_model = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=60,
        max_depth=16,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ))
])

print('Fitting LinearRegression...')
lr_model.fit(X_train, y_train)
print('Fitting RandomForest...')
rf_model.fit(X_train, y_train)

best_model = rf_model
best_name = 'RandomForest'
print('Best model selected:', best_name)

pred_lr = lr_model.predict(X_val)
pred_rf = rf_model.predict(X_val)
pred_adv = pred_rf

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = {
    'LinearRegression': {'mae': mae(y_val, pred_lr), 'rmse': rmse(y_val, pred_lr)},
    'RandomForest': {'mae': mae(y_val, pred_rf), 'rmse': rmse(y_val, pred_rf)},
}

for name, metrics in results.items():
    print(f"{name}: MAE={metrics['mae']:.3f}, RMSE={metrics['rmse']:.3f}")

print('Best on MAE:', best_name)

In [ ]:
# ==============================
# 8) Feature Importance and Diagnostics
# ==============================

# Extract feature names from the preprocessing pipeline
prepared_feature_names = best_model.named_steps['prep'].get_feature_names_out()
model_estimator = best_model.named_steps['model']

if hasattr(model_estimator, 'feature_importances_'):
    fi = pd.DataFrame({
        'feature': prepared_feature_names,
        'importance': model_estimator.feature_importances_
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(10, 7))
    sns.barplot(x='importance', y='feature', data=fi.head(20))
    plt.title('Tree Model Feature Importance')
    plt.tight_layout()
    plt.show()
else:
    fi = pd.DataFrame({'feature': prepared_feature_names, 'importance': np.nan})

plt.figure(figsize=(12, 5))
plt.plot(y_val.reset_index(drop=True), label='Actual')
plt.plot(pd.Series(pred_adv).reset_index(drop=True), label='Predicted')
plt.legend()
plt.title('Actual vs Predicted (Validation)')
plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 9) Save Model, Validation Forecasts, and Submission File
# ==============================

os.makedirs('artifacts', exist_ok=True)
joblib.dump(best_model, 'artifacts/best_model.joblib')

validation_output = pd.DataFrame({
    'actual_num_orders': y_val.values,
    'pred_num_orders': pred_adv
})
validation_output.to_csv('artifacts/validation_forecast.csv', index=False)


def build_test_features(train_history, future_frame):
    history = train_history.sort_values(['center_id', 'meal_id', 'week']).copy()
    future = future_frame.copy().sort_values(['center_id', 'meal_id', 'week']).reset_index(drop=True)

    global_order_median = float(history['num_orders'].median())
    global_price_median = float(history['checkout_price'].median())
    center_means = history.groupby('center_id')['num_orders'].mean().to_dict()
    meal_means = history.groupby('meal_id')['num_orders'].mean().to_dict()

    rows = []
    for _, row in future.iterrows():
        group_history = history[(history['center_id'] == row['center_id']) & (history['meal_id'] == row['meal_id'])].sort_values('week')
        past_orders = group_history['num_orders'].tolist()
        past_prices = group_history['checkout_price'].tolist()

        feature_row = row.to_dict()
        feature_row['price_diff'] = feature_row['checkout_price'] - feature_row['base_price']
        feature_row['price_discount_pct'] = ((feature_row['base_price'] - feature_row['checkout_price']) / feature_row['base_price']) if feature_row['base_price'] else 0
        feature_row['is_discount'] = int(feature_row['price_discount_pct'] > 0)
        feature_row['price_ratio'] = (feature_row['checkout_price'] / feature_row['base_price']) if feature_row['base_price'] else 1

        for lag in [1, 2, 3, 4]:
            feature_row[f'lag_orders_{lag}'] = past_orders[-lag] if len(past_orders) >= lag else global_order_median
            feature_row[f'lag_checkout_price_{lag}'] = past_prices[-lag] if len(past_prices) >= lag else global_price_median

        recent_orders = past_orders[-4:] if len(past_orders) >= 4 else past_orders
        recent_orders_8 = past_orders[-8:] if len(past_orders) >= 8 else past_orders
        feature_row['rolling_mean_4'] = float(np.mean(recent_orders)) if recent_orders else global_order_median
        feature_row['rolling_std_4'] = float(np.std(recent_orders, ddof=1)) if len(recent_orders) > 1 else 0.0
        feature_row['rolling_mean_8'] = float(np.mean(recent_orders_8)) if recent_orders_8 else global_order_median
        feature_row['week_sin'] = float(np.sin(2 * np.pi * feature_row['week'] / 52.0))
        feature_row['week_cos'] = float(np.cos(2 * np.pi * feature_row['week'] / 52.0))
        feature_row['center_week_avg'] = float(center_means.get(feature_row['center_id'], global_order_median))
        feature_row['meal_week_avg'] = float(meal_means.get(feature_row['meal_id'], global_order_median))
        rows.append(feature_row)

    return pd.DataFrame(rows)

# Build submission features from the train history and predict the provided test set.
test_features = build_test_features(train_df, test_df)
submission_predictions = best_model.predict(test_features[feature_cols])

if 'sample_submission' in globals() and 'id' in sample_submission.columns:
    submission_ids = sample_submission['id']
elif 'id' in test_raw.columns:
    submission_ids = test_raw['id']
else:
    submission_ids = pd.Series(range(len(test_raw)))

submission = pd.DataFrame({
    'id': submission_ids,
    'num_orders': np.maximum(0, submission_predictions)
})
submission.to_csv('artifacts/sample_submission_completed.csv', index=False)

print('Saved model, validation_forecast.csv, and sample_submission_completed.csv to artifacts/')

# Multi-step forecasting and explainability are adapted to the weekly dataset.
# The trained model can be used for validation, direct test predictions, and submission generation.

# 10) Optional future-week scenario simulation and SHAP analysis are included below.

In [ ]:
# ======== Direct multi-week scenario simulation ========

# Create a simple future scenario by extending the latest observed rows with stable weekly averages.
latest_week = int(train_df['week'].max())
latest_center_meal = train_df[['center_id', 'meal_id', 'category', 'cuisine', 'city_code', 'region_code', 'center_type']].drop_duplicates()

# Cap scenario size for faster execution on large datasets.
MAX_SCENARIO_ENTRIES = 2000
if len(latest_center_meal) > MAX_SCENARIO_ENTRIES:
    latest_center_meal = latest_center_meal.sample(MAX_SCENARIO_ENTRIES, random_state=42)

median_checkout = float(train_df['checkout_price'].median())
median_base = float(train_df['base_price'].median())
median_orders = float(train_df['num_orders'].median())
std_orders = float(train_df['num_orders'].std())
center_means = train_df.groupby('center_id')['num_orders'].mean()
meal_means = train_df.groupby('meal_id')['num_orders'].mean()

scenario_rows = []
for horizon_week in range(1, 8):
    future_week = latest_week + horizon_week
    for base_row in latest_center_meal.itertuples(index=False):
        scenario_rows.append({
            'center_id': base_row.center_id,
            'meal_id': base_row.meal_id,
            'checkout_price': median_checkout,
            'base_price': median_base,
            'emailer_for_promotion': 0,
            'homepage_featured': 0,
            'category': base_row.category,
            'cuisine': base_row.cuisine,
            'city_code': base_row.city_code,
            'region_code': base_row.region_code,
            'center_type': base_row.center_type,
            'week': future_week,
            'price_diff': median_checkout - median_base,
            'price_discount_pct': 0.0,
            'is_discount': 0,
            'price_ratio': 1.0,
            'lag_orders_1': median_orders,
            'lag_orders_2': median_orders,
            'lag_orders_3': median_orders,
            'lag_orders_4': median_orders,
            'lag_checkout_price_1': median_checkout,
            'lag_checkout_price_2': median_checkout,
            'lag_checkout_price_3': median_checkout,
            'lag_checkout_price_4': median_checkout,
            'rolling_mean_4': median_orders,
            'rolling_std_4': std_orders,
            'rolling_mean_8': median_orders,
            'week_sin': np.sin(2 * np.pi * future_week / 52.0),
            'week_cos': np.cos(2 * np.pi * future_week / 52.0),
            'center_week_avg': float(center_means.get(base_row.center_id, median_orders)),
            'meal_week_avg': float(meal_means.get(base_row.meal_id, median_orders)),
        })

scenario_df = pd.DataFrame(scenario_rows)
scenario_predictions = best_model.predict(scenario_df[feature_cols])
scenario_df['predicted_num_orders'] = scenario_predictions
scenario_df[['center_id', 'meal_id', 'week', 'predicted_num_orders']].head()

In [ ]:
# ======== SHAP Explainability for the weekly model ========

# Use a transformed sample for SHAP on the tree model if the transformed matrix is supported.
transformed_val = best_model.named_steps['prep'].transform(X_val)
feature_names = best_model.named_steps['prep'].get_feature_names_out()

if hasattr(transformed_val, 'toarray'):
    transformed_val = transformed_val.toarray()

if hasattr(model_estimator, 'predict'):
    try:
        explainer = shap.Explainer(model_estimator, transformed_val)
        shap_values = explainer(transformed_val)

        shap_importance = pd.DataFrame({
            'feature': feature_names,
            'mean_abs_shap': np.abs(shap_values.values).mean(axis=0)
        }).sort_values('mean_abs_shap', ascending=False)

        print('Top SHAP drivers:')
        print(shap_importance.head(10))

        shap_importance.to_csv('artifacts/shap_feature_contributions.csv', index=False)

        plt.figure(figsize=(9, 6))
        sns.barplot(data=shap_importance.head(20), x='mean_abs_shap', y='feature', color='#2a9d8f')
        plt.title('Global SHAP Feature Contributions')
        plt.tight_layout()
        plt.savefig('artifacts/shap_global_contributions.png', dpi=180, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print('SHAP analysis skipped:', e)
else:
    print('Model does not support SHAP analysis in this notebook.')

In [ ]:
# ==============================
# 11) Week 4 Summary and Business Output
# ==============================

summary = {
    'rows_train': int(train_df.shape[0]),
    'rows_model': int(model_df.shape[0]),
    'validation_mae': float(results[best_name]['mae']),
    'validation_rmse': float(results[best_name]['rmse']),
    'best_model': best_name,
    'top_weekly_features': fi.head(10)['feature'].tolist() if 'fi' in globals() and 'feature' in fi.columns else [],
}

print('Week 3/4 summary:')
for key, value in summary.items():
    print(f'{key}: {value}')

summary_df = pd.DataFrame([summary])
summary_df.to_csv('artifacts/week3_week4_summary.csv', index=False)
print('Saved artifacts/week3_week4_summary.csv')